In [2]:
import pandas as pd
import numpy as np

p_train, p_test, p_val = 0.8, 0.1, 0.1
XLSX_PATH = "../data/solar_panel_data_madagascar.xlsx"
OUTPUT_PATH = "../data"
seed = 56

images, elements, coordinates = pd.read_excel(XLSX_PATH, sheet_name=[0, 1, 2]).values()

In [3]:
coordinates = coordinates[pd.to_numeric(coordinates["lat"], errors="coerce").notnull()]
coordinates = coordinates[pd.to_numeric(coordinates["long"], errors="coerce").notnull()]
coordinates["lat"] = coordinates["lat"].astype(float)
coordinates["long"] = coordinates["long"].astype(float)
coordinates

,Unnamed: 0,elt_name,edge_rank,long,lat
0,0,2z1,1.0,212.0,116.0
1,1,2z1,2.0,316.0,164.0
2,2,2z1,3.0,284.0,244.0
3,3,2z1,4.0,178.0,205.0
4,4,3z1,1.0,153.0,201.0
...,...,...,...,...,...
97184,97205,2300z24,4.0,3037.0,561.0
97185,97206,2300z25,1.0,3239.0,207.0
97186,97207,2300z25,2.0,3341.0,173.0
97187,97208,2300z25,3.0,3415.0,479.0


In [4]:
elements = elements[elements["type1"] == "pan"]
elements = elements[elements["elt_name"].isin(coordinates["elt_name"])]
elements

,Unnamed: 0,img_name,elt_name,type1,type2,boil_nbr,pan_nbr,pan_area_sqm,notes
0,0,953,953z2,pan,roof,NaN,1.0,0.65,NaN
1,1,691,691z1,pan,roof,NaN,1.0,0.68,NaN
2,2,705,705z2,pan,roof,NaN,1.0,0.68,NaN
3,3,593,593z1,pan,roof,NaN,1.0,0.75,NaN
4,4,593,593z2,pan,roof,NaN,1.0,0.75,NaN
...,...,...,...,...,...,...,...,...,...
22478,22483,2300ID,2300z21,pan,NaN,NaN,36.0,NaN,NaN
22479,22484,2300ID,2300z22,pan,NaN,NaN,36.0,NaN,NaN
22480,22485,2300ID,2300z23,pan,NaN,NaN,36.0,NaN,NaN
22481,22486,2300ID,2300z24,pan,NaN,NaN,36.0,NaN,NaN


In [5]:
images = images[images["img_origin"] == "D"]
images = images[images["type1"] != "boil"]
images = images[images["img_name"].isin(elements["img_name"])]
images

,Unnamed: 0,number,img_name,img_long_East,img_lat_South,img_origin,type1,elt_nbr,type2,park_capacity_kw,...,mouse_alt,width_pixel,height_pixel,city,region,img_date,img_time,img_altsea,zoom,img_ISO
2060,2060,2126,2126ID,43.574062,-23.048845,D,pan,4,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:46:05,49.29,1,100.0
2061,2061,2128,2128ID,43.569401,-23.046642,D,pan,2,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:49:54,83.18,6.3,100.0
2062,2062,2129,2129ID,43.567268,-23.045612,D,mix,7,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:51:26,83.07,2.86,100.0
2063,2063,2130,2130ID,43.567265,-23.045612,D,mix,5,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:52:01,83.00,2.86,100.0
2064,2064,2132,2132ID,43.574135,-23.046333,D,pan,20,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,07:33:54,73.30,1.26,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11123,11131,11323,11323ID,46.745594,-16.138494,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:57:53,94.53,1.64,100.0
11124,11132,11324,11324ID,46.746655,-16.137222,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:29,90.06,2.58,110.0
11125,11133,11325,11325ID,46.747299,-16.136864,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:45,83.13,1.27,110.0
11126,11134,11326,11326ID,46.747299,-16.136864,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:55,82.66,2.32,100.0


In [6]:
grouped_coordinates = coordinates.groupby("elt_name").agg({"long": list, "lat": list})
bounding_boxes = grouped_coordinates.apply(
    lambda row: [min(row["long"]), min(row["lat"]), max(row["long"]), max(row["lat"])], axis=1
)

In [7]:
bounding_boxes

elt_name
10000z1    [1475.0, 1629.0, 2825.0, 2526.0]
10001z1    [2459.0, 1791.0, 3146.0, 2445.0]
10001z2    [2852.0, 1239.0, 3019.0, 1422.0]
10002z1     [983.0, 1306.0, 2213.0, 1962.0]
10002z2    [1063.0, 2109.0, 2583.0, 3076.0]
                         ...               
999z1          [539.0, 252.0, 596.0, 304.0]
99z1           [156.0, 138.0, 291.0, 261.0]
99z2           [161.0, 280.0, 363.0, 492.0]
9z1             [87.0, 116.0, 205.0, 186.0]
9z2            [151.0, 541.0, 193.0, 581.0]
Length: 22461, dtype: object

In [8]:
grouped_bounding_boxes = bounding_boxes.groupby("elt_name").agg(bounding_boxes=list).reset_index()
elements_with_grouped_bounding_boxes = pd.merge(
    elements, grouped_bounding_boxes, on="elt_name", how="left"
)
elements_with_grouped_bounding_boxes

,Unnamed: 0,img_name,elt_name,type1,type2,boil_nbr,pan_nbr,pan_area_sqm,notes,bounding_boxes
0,0,953,953z2,pan,roof,NaN,1.0,0.65,NaN,"[[204.0, 129.0, 278.0, 207.0]]"
1,1,691,691z1,pan,roof,NaN,1.0,0.68,NaN,"[[179.0, 138.0, 237.0, 199.0]]"
2,2,705,705z2,pan,roof,NaN,1.0,0.68,NaN,"[[365.0, 332.0, 410.0, 393.0]]"
3,3,593,593z1,pan,roof,NaN,1.0,0.75,NaN,"[[87.0, 181.0, 99.0, 214.0]]"
4,4,593,593z2,pan,roof,NaN,1.0,0.75,NaN,"[[111.0, 187.0, 127.0, 220.0]]"
...,...,...,...,...,...,...,...,...,...,...
19960,22483,2300ID,2300z21,pan,NaN,NaN,36.0,NaN,NaN,"[[2119.0, 189.0, 2281.0, 595.0]]"
19961,22484,2300ID,2300z22,pan,NaN,NaN,36.0,NaN,NaN,"[[2441.0, 179.0, 2561.0, 581.0]]"
19962,22485,2300ID,2300z23,pan,NaN,NaN,36.0,NaN,NaN,"[[2705.0, 175.0, 2857.0, 585.0]]"
19963,22486,2300ID,2300z24,pan,NaN,NaN,36.0,NaN,NaN,"[[2973.0, 173.0, 3137.0, 561.0]]"


In [9]:
grouped_elements_with_grouped_bounding_boxes = (
    elements_with_grouped_bounding_boxes.groupby("img_name")["bounding_boxes"]
    .agg(list_bounding_boxes="sum")
    .reset_index()
)
images_with_list_bounding_boxes = pd.merge(
    images, grouped_elements_with_grouped_bounding_boxes, on="img_name", how="left"
)
images_with_list_bounding_boxes = images_with_list_bounding_boxes[
    ["img_name", "list_bounding_boxes"]
]
images_with_list_bounding_boxes

,img_name,list_bounding_boxes
0,2126ID,"[[2285.0, 2518.0, 2418.0, 2626.0], [2310.0, 25..."
1,2128ID,"[[2368.0, 1712.0, 2872.0, 2128.0], [2768.0, 18..."
2,2129ID,"[[832.0, 2880.0, 2008.0, 3608.0], [960.0, 2320..."
3,2130ID,"[[1044.0, 2364.0, 2172.0, 2880.0], [984.0, 256..."
4,2132ID,"[[1736.0, 1588.0, 2132.0, 1680.0], [1752.0, 16..."
...,...,...
8972,11323ID,"[[1732.0, 1356.0, 2060.0, 1568.0]]"
8973,11324ID,"[[1905.0, 1344.0, 2418.0, 1476.0]]"
8974,11325ID,"[[2249.0, 1521.0, 2570.0, 1635.0]]"
8975,11326ID,"[[1194.0, 1350.0, 1452.0, 1512.0]]"


In [10]:
N = len(images_with_list_bounding_boxes)
N_train = int(p_train * N)
N_test = int(p_test * N)
shuffled_img_names = images_with_list_bounding_boxes.sample(frac=1, random_state=seed).reset_index(
    drop=True
)
train_images, test_images, val_images = (
    shuffled_img_names[:N_train].reset_index(drop=True),
    shuffled_img_names[N_train : N_train + N_test].reset_index(drop=True),
    shuffled_img_names[N_train + N_test :].reset_index(drop=True),
)

In [11]:
train_images.to_json(OUTPUT_PATH + "/train_images.json", index=False)
test_images.to_json(OUTPUT_PATH + "/test_images.json", index=False)
val_images.to_json(OUTPUT_PATH + "/val_images.json", index=False)

In [ ]:
import pathlib

for file in pathlib.Path("../data/img").iterdir():
    if file.name.split(".")[0] not in images_with_list_bounding_boxes["img_name"].values:
        file.unlink()